# 02 · Comprensión inicial de la fuente

> **ALCANCE HISTÓRICO DEL ESCENARIO TOTAL.** Este perfilado de la fuente completa
> no constituye un EDA territorial/temporal equivalente al escenario vigente.
> No se completa ese EDA porque los registros `Solo Daños` sufren un cambio de
> cobertura; el escenario total se conserva solo como sensibilidad histórica.
> El EDA canónico es 02B y usa exclusivamente siniestros con víctimas.

**Actualización: 1 de octubre de 2026.** Perfilado descriptivo, de solo lectura,
de las cuatro hojas del Excel congelado del Anuario de Siniestralidad 2024.
Se explora la fuente completa (2015–2024), sin aplicar aquí el filtro de víctimas
ni preparar variables del modelo.

Este notebook permite conocer dimensiones, columnas, tipos inferidos por pandas,
nulos y duplicados. Una columna numérica no necesariamente es una magnitud:
por ejemplo, los códigos identifican registros y su promedio no tiene una
interpretación sustantiva. Los tipos de pandas no sustituyen el diccionario ni
las validaciones semánticas de preparación.

El EDA vigente de siniestros con víctimas de 2018–2024 está en
[02B_EDA_y_Analisis_Espacial_Con_Victimas.ipynb](02B_EDA_y_Analisis_Espacial_Con_Victimas.ipynb).
La [ficha de procedencia](../data/raw/procedencia_excel.json) conserva el hash,
dimensiones, usos y limitaciones de la fuente. No se deben sumar las filas de
Siniestros, Vehículos y Actor_vial como si representaran la misma unidad.

La ejecución lee el Excel local, comprueba su identidad y conserva las salidas
de exploración. No escribe datasets ni entrena modelos.


In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import display

ROOT = Path(os.environ['SINIESTRALIDAD_ROOT']).expanduser().resolve() if 'SINIESTRALIDAD_ROOT' in os.environ else next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/preparacion.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Abra desde la raíz/notebooks o defina SINIESTRALIDAD_ROOT.')
sys.path.insert(0, str(ROOT))
from src.rutas import sha256_archivo

RAW_FILE = ROOT / 'data/raw/base-anuario-de-siniestralidad-2024.xlsx'
print('Entorno de esta ejecución: Python', sys.version.split()[0], '| pandas', pd.__version__)


Entorno de esta ejecución: Python 3.12.14 | pandas 3.0.3


In [2]:
# Lectura de la copia congelada; no se descarga ni modifica el archivo fuente.
procedencia = json.loads((ROOT / 'data/raw/procedencia_excel.json').read_text())
assert sha256_archivo(RAW_FILE) == procedencia['sha256_local'], 'El Excel no coincide con la fuente registrada.'
hojas = pd.read_excel(RAW_FILE, sheet_name=None)
assert set(hojas) == set(procedencia['hojas']), 'Hojas distintas de las registradas.'
for nombre, tabla in hojas.items():
    esperado = procedencia['hojas'][nombre]
    assert tabla.shape == (esperado['filas'], esperado['columnas']), f'Dimensiones inesperadas: {nombre}'
siniestros = hojas['Siniestros']
vehiculos = hojas['Vehiculos']
Actor_vial = hojas['Actor_vial']
Diccionario = hojas['Diccionario']
print('Fuente verificada. Hojas cargadas:', ', '.join(hojas))


Fuente verificada. Hojas cargadas: Siniestros, Vehiculos, Actor_vial, Diccionario


# Información 1. Siniestros

En esta sección se realiza la exploración inicial del conjunto de datos de **Siniestros Viales** correspondiente al Anuario de Siniestralidad de Bogotá. El objetivo es comprender la estructura de la información, identificar las variables disponibles, analizar la calidad de los datos y obtener una visión general de los registros que servirán como base para el análisis exploratorio y el desarrollo de modelos predictivos.

In [3]:
# Muestra las primeras 5 filas del DataFrame para conocer la estructura de los datos.
siniestros.head()

,Codigo_Accidente,Formulario,Longitud,Latitud,Direccion,Fecha_Acc,AA_Acc,MM_Acc,DD_Mes_Acc,Dia_Semana_Acc,...,Con_Sitp,Con_Troncal,Con_Alimentador,Con_Zonal,Con_Provisional,Con_Articulado,Con_Biarticulado,Con_Padron_Dual,Con_Servicio_Especial,Con_Taxi
0,4404210,A685,-74.181387,4.621978,AV AVENIDA CIUDAD DE CALI-KR 55 S 02,2015-02-09,2015,Febrero,9,lunes,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4404211,A666,-74.027304,4.762901,AK 7-CL 186 02,2015-02-06,2015,Febrero,6,viernes,...,SI,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN,NaN
2,4404212,A698,-74.053529,4.706939,KR 45-CL 127 02,2015-02-07,2015,Febrero,7,sábado,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4404213,A000042495,-74.099563,4.576256,KR 12-CL 24 S 06,2015-02-09,2015,Febrero,9,lunes,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4404214,A644,-74.091667,4.524080,CL 70-KR 11D SE 04,2015-02-08,2015,Febrero,8,domingo,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Devuelve una tupla con el número de filas y columnas (filas, columnas).
siniestros.shape

(278614, 40)

In [5]:
# Muestra los nombres de las columnas; no el número de registros.
siniestros.columns


Index(['Codigo_Accidente', 'Formulario', 'Longitud', 'Latitud', 'Direccion',
       'Fecha_Acc', 'AA_Acc', 'MM_Acc', 'DD_Mes_Acc', 'Dia_Semana_Acc',
       'Hora_Acc', 'Min_Acc', 'Localidad', 'Clase_Acc', 'Elemento_Choque',
       'Tipo_Objeto_Fijo', 'Gravedad_Indicador_Tradicional',
       'Gravedad_indicador_30d', 'Con_Bicicleta', 'Con_Carga',
       'Con_Embriaguez', 'Con_Huecos', 'Con_Menores', 'Con_Moto', 'Con_Peaton',
       'Con_Persona_Mayor', 'Con_Rutas', 'Con_Tpi', 'Con_Tpp', 'Con_Velocidad',
       'Con_Sitp', 'Con_Troncal', 'Con_Alimentador', 'Con_Zonal',
       'Con_Provisional', 'Con_Articulado', 'Con_Biarticulado',
       'Con_Padron_Dual', 'Con_Servicio_Especial', 'Con_Taxi'],
      dtype='str')

In [6]:
# Invoca info(): muestra todas las columnas, tipos y conteos no nulos.
# show_counts=True evita que pandas omita los conteos por el tamaño de la hoja.
siniestros.info(verbose=True, show_counts=True)


<class 'pandas.DataFrame'>
RangeIndex: 278614 entries, 0 to 278613
Data columns (total 40 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   Codigo_Accidente                278614 non-null  int64         
 1   Formulario                      278614 non-null  str           
 2   Longitud                        263549 non-null  float64       
 3   Latitud                         263549 non-null  float64       
 4   Direccion                       278614 non-null  str           
 5   Fecha_Acc                       278614 non-null  datetime64[us]
 6   AA_Acc                          278614 non-null  int64         
 7   MM_Acc                          278614 non-null  str           
 8   DD_Mes_Acc                      278614 non-null  int64         
 9   Dia_Semana_Acc                  278614 non-null  str           
 10  Hora_Acc                        278614 non-null  int64         
 11

In [7]:
# Genera estadísticas descriptivas de las variables numéricas
# (conteo, promedio, desviación estándar, mínimo, máximo, etc.).
siniestros.describe()

,Codigo_Accidente,Longitud,Latitud,Fecha_Acc,AA_Acc,DD_Mes_Acc,Hora_Acc,Min_Acc
count,2.786140e+05,263549.000000,263549.000000,278614,278614.000000,278614.000000,278614.000000,278614.000000
mean,8.055555e+06,-74.105452,4.647105,2019-04-23 22:38:26.520132,2018.803588,15.783920,12.902133,24.677633
min,4.401419e+06,-74.247270,4.043953,2015-01-01 00:00:00,2015.000000,1.000000,0.000000,0.000000
25%,4.472503e+06,-74.135597,4.606198,2017-02-10 00:00:00,2017.000000,8.000000,9.000000,10.000000
50%,1.047282e+07,-74.104129,4.641827,2019-01-13 00:00:00,2019.000000,16.000000,13.000000,28.000000
75%,1.054390e+07,-74.074978,4.688928,2021-07-04 00:00:00,2021.000000,23.000000,17.000000,40.000000
max,1.061630e+07,-74.007000,4.828041,2024-12-31 00:00:00,2024.000000,31.000000,23.000000,59.000000
std,2.983755e+06,0.040043,0.057906,NaN,2.628660,8.777524,5.508462,17.420744


In [8]:
# Cuenta la cantidad de valores nulos en cada columna.
siniestros.isnull().sum()

Codigo_Accidente                       0
Formulario                             0
Longitud                           15065
Latitud                            15065
Direccion                              0
Fecha_Acc                              0
AA_Acc                                 0
MM_Acc                                 0
DD_Mes_Acc                             0
Dia_Semana_Acc                         0
Hora_Acc                               0
Min_Acc                                0
Localidad                              0
Clase_Acc                              0
Elemento_Choque                    44904
Tipo_Objeto_Fijo                  268114
Gravedad_Indicador_Tradicional         0
Gravedad_indicador_30d               148
Con_Bicicleta                     256804
Con_Carga                         241511
Con_Embriaguez                    272688
Con_Huecos                        276298
Con_Menores                       263921
Con_Moto                          194012
Con_Peaton      

In [9]:
# Cuenta el número de registros duplicados.
siniestros.duplicated().sum()

np.int64(0)

# Información 2. Vehículos

Exploración inicial de la hoja de vehículos involucrados en los siniestros. Se
revisan estructura, tipos, nulos y duplicados. Un siniestro puede tener varios
vehículos: estos registros no deben contarse como siniestros adicionales.

Esta hoja no aporta predictores al modelo principal actual. El análisis
descriptivo del escenario con víctimas se documenta en 02B.


In [10]:
 # Muestra las primeras 5 filas del DataFrame para conocer la estructura de los datos.
vehiculos.head()

,Codigo_Accidente,Formulario,Fecha_Acc,AA_Acc,Codigo_Vehiculo,Clase,Servicio,Modalidad,Vehiculo_Viajaba_Clasificado,Tipo_SITP,...,Con_Sitp,Con_Troncal,Con_Alimentador,Con_Zonal,Con_Provisional,Con_Articulado,Con_Biarticulado,Con_Padron_Dual,Con_Servicio_Especial,Con_Taxi
0,4401718,A000040450,2015-01-07,2015,2.0,Motocicleta,Particular,NaN,MOTOCICLETA,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4401718,A000040450,2015-01-07,2015,3.0,NaN,Sin información,NaN,SIN INFORMACIÓN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4401726,A000040954,2015-01-07,2015,1.0,Bus,Público,NaN,TRANSPORTE DE PASAJEROS,BIARTICULADO,...,SI,SI,NaN,NaN,NaN,NaN,SI,NaN,NaN,NaN
3,4401725,A000040171,2015-01-07,2015,1.0,Automovil,Particular,NaN,LIVIANO,NaN,...,SI,SI,NaN,NaN,NaN,NaN,NaN,SI,NaN,NaN
4,4401725,A000040171,2015-01-07,2015,2.0,Bus,Público,NaN,TRANSPORTE DE PASAJEROS,PADRON DUAL,...,SI,SI,NaN,NaN,NaN,NaN,NaN,SI,NaN,NaN


In [11]:
 # Devuelve una tupla con el número de filas y columnas (filas, columnas).
vehiculos.shape

(523168, 32)

In [12]:

# Muestra los nombres de las columnas del DataFrame.
vehiculos.columns

Index(['Codigo_Accidente', 'Formulario', 'Fecha_Acc', 'AA_Acc',
       'Codigo_Vehiculo', 'Clase', 'Servicio', 'Modalidad',
       'Vehiculo_Viajaba_Clasificado', 'Tipo_SITP', 'Con_Bicicleta',
       'Con_Carga', 'Con_Embriaguez', 'Con_Huecos', 'Con_Menores', 'Con_Moto',
       'Con_Peaton', 'Con_Persona_Mayor', 'Con_Rutas', 'Con_Tpi', 'Con_Tpp',
       'Con_Velocidad', 'Con_Sitp', 'Con_Troncal', 'Con_Alimentador',
       'Con_Zonal', 'Con_Provisional', 'Con_Articulado', 'Con_Biarticulado',
       'Con_Padron_Dual', 'Con_Servicio_Especial', 'Con_Taxi'],
      dtype='str')

In [13]:
# Invoca info(): muestra todas las columnas, tipos y conteos no nulos.
# show_counts=True evita que pandas omita los conteos por el tamaño de la hoja.
vehiculos.info(verbose=True, show_counts=True)


<class 'pandas.DataFrame'>
RangeIndex: 523168 entries, 0 to 523167
Data columns (total 32 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   Codigo_Accidente              523168 non-null  int64         
 1   Formulario                    523168 non-null  str           
 2   Fecha_Acc                     523168 non-null  datetime64[us]
 3   AA_Acc                        523168 non-null  int64         
 4   Codigo_Vehiculo               523165 non-null  float64       
 5   Clase                         519145 non-null  str           
 6   Servicio                      523168 non-null  str           
 7   Modalidad                     101939 non-null  str           
 8   Vehiculo_Viajaba_Clasificado  523168 non-null  str           
 9   Tipo_SITP                     47561 non-null   str           
 10  Con_Bicicleta                 43398 non-null   str           
 11  Con_Carga               

In [14]:
 # Genera estadísticas descriptivas de las variables numéricas
# (conteo, promedio, desviación estándar, mínimo, máximo, etc.).
vehiculos.describe()

,Codigo_Accidente,Fecha_Acc,AA_Acc,Codigo_Vehiculo
count,5.231680e+05,523168,523168.000000,523165.000000
mean,8.032376e+06,2019-04-09 07:34:39.760229,2018.763512,1.510111
min,4.401419e+06,2015-01-01 00:00:00,2015.000000,1.000000
25%,4.472138e+06,2017-02-07 00:00:00,2017.000000,1.000000
50%,1.047159e+07,2018-12-26 00:00:00,2018.000000,1.000000
75%,1.054193e+07,2021-06-09 00:00:00,2021.000000,2.000000
max,1.061630e+07,2024-12-31 00:00:00,2024.000000,30.000000
std,2.987301e+06,NaN,2.599862,0.626443


In [15]:
 # Cuenta la cantidad de valores nulos en cada columna.
vehiculos.isnull().sum()

Codigo_Accidente                     0
Formulario                           0
Fecha_Acc                            0
AA_Acc                               0
Codigo_Vehiculo                      3
Clase                             4023
Servicio                             0
Modalidad                       421229
Vehiculo_Viajaba_Clasificado         0
Tipo_SITP                       475607
Con_Bicicleta                   479770
Con_Carga                       447986
Con_Embriaguez                  511971
Con_Huecos                      519729
Con_Menores                     499741
Con_Moto                        365445
Con_Peaton                      487801
Con_Persona_Mayor               441987
Con_Rutas                       522988
Con_Tpi                         414660
Con_Tpp                         398929
Con_Velocidad                   518994
Con_Sitp                        438368
Con_Troncal                     516598
Con_Alimentador                 515774
Con_Zonal                

In [16]:
 # Cuenta el número de registros duplicados.
vehiculos.duplicated().sum()

np.int64(0)

# Información 3. Actores Viales

Exploración inicial de la hoja de actores involucrados en los siniestros:
peatones, motociclistas, ciclistas, conductores y pasajeros. Se revisan su
estructura y calidad; sus filas no equivalen al número de siniestros.

Los actores se utilizan en análisis descriptivos de 02B, no como predictores
ni para emitir alertas individuales o específicas por actor en el modelo actual.


In [17]:
# Muestra las primeras 5 filas del DataFrame para conocer la estructura de los datos.
Actor_vial.head()

,Codigo_Accidentado,Codigo_Accidente,Formulario,Codigo_Vehiculo,Ccodigo_Victima,FechaAcc,AnnoAcc,mesAcc,DD_Mes_Acc,Dia_Semana_Acc,...,Con_Sitp,Con_Troncal,Con_Alimentador,Con_Zonal,Con_Provisional,Con_Articulado,Con_Biarticulado,Con_Padron_Dual,Con_Servicio_Especial,Con_Taxi
0,2456665,4403353,A000041152,2.0,0,2015-01-28,2015,Enero,28,miércoles,...,SI,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN,SI
1,2456666,4403354,A000042570,1.0,0,2015-01-28,2015,Enero,28,miércoles,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2456667,4403354,A000042570,2.0,0,2015-01-28,2015,Enero,28,miércoles,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2456670,4403356,A000042404,1.0,0,2015-01-28,2015,Enero,28,miércoles,...,SI,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN,NaN
4,2456671,4403356,A000042404,2.0,0,2015-01-28,2015,Enero,28,miércoles,...,SI,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# Devuelve una tupla con el número de filas y columnas (filas, columnas).
Actor_vial.shape

(608155, 44)

In [19]:
# Lista los nombres de todas las columnas del DataFrame.
Actor_vial.columns

Index(['Codigo_Accidentado', 'Codigo_Accidente', 'Formulario',
       'Codigo_Vehiculo', 'Ccodigo_Victima', 'FechaAcc', 'AnnoAcc', 'mesAcc',
       'DD_Mes_Acc', 'Dia_Semana_Acc', 'hourAcc', 'minuAcc', 'Localidad',
       'Edad', 'Sexo', 'Gravedad_Indicador_Tradicional', 'Muerte_Posterior',
       'Fecha_CambioGravedad', 'Gravedad_Indicador_30d', 'Condicion',
       'Condicion_Especifica', 'Tipo_SITP', 'Con_Bicicleta', 'Con_Carga',
       'Con_Embriaguez', 'Con_Huecos', 'Con_Menores', 'Con_Moto', 'Con_Peaton',
       'Con_Persona_Mayor', 'Con_Rutas', 'Con_Tpi', 'Con_Tpp', 'Con_Velocidad',
       'Con_Sitp', 'Con_Troncal', 'Con_Alimentador', 'Con_Zonal',
       'Con_Provisional', 'Con_Articulado', 'Con_Biarticulado',
       'Con_Padron_Dual', 'Con_Servicio_Especial', 'Con_Taxi'],
      dtype='str')

In [20]:
# Invoca info(): muestra todas las columnas, tipos y conteos no nulos.
# show_counts=True evita que pandas omita los conteos por el tamaño de la hoja.
Actor_vial.info(verbose=True, show_counts=True)


<class 'pandas.DataFrame'>
RangeIndex: 608155 entries, 0 to 608154
Data columns (total 44 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   Codigo_Accidentado              608155 non-null  int64         
 1   Codigo_Accidente                608155 non-null  int64         
 2   Formulario                      608155 non-null  str           
 3   Codigo_Vehiculo                 595469 non-null  float64       
 4   Ccodigo_Victima                 608155 non-null  int64         
 5   FechaAcc                        608155 non-null  datetime64[us]
 6   AnnoAcc                         608155 non-null  int64         
 7   mesAcc                          608155 non-null  str           
 8   DD_Mes_Acc                      608155 non-null  int64         
 9   Dia_Semana_Acc                  608155 non-null  str           
 10  hourAcc                         608155 non-null  int64         
 11

In [21]:
# Genera estadísticas descriptivas de las variables numéricas
# (conteo, promedio, desviación estándar, mínimo, máximo, etc.).
Actor_vial.describe()

,Codigo_Accidentado,Codigo_Accidente,Codigo_Vehiculo,Ccodigo_Victima,FechaAcc,AnnoAcc,DD_Mes_Acc,hourAcc,minuAcc,Edad,Fecha_CambioGravedad
count,6.081550e+05,6.081550e+05,595469.000000,608155.000000,608155,608155.000000,608155.000000,608155.000000,608155.000000,594818.000000,1920
mean,8.648353e+06,8.089408e+06,1.442426,0.389809,2019-05-14 07:19:13.949896,2018.858982,15.784164,12.904763,24.758989,37.557703,2019-12-14 14:04:30
min,2.452497e+06,4.401419e+06,0.000000,0.000000,2015-01-01 00:00:00,2015.000000,1.000000,0.000000,0.000000,0.000000,2015-01-10 00:00:00
25%,2.606742e+06,4.473416e+06,1.000000,0.000000,2017-02-18 00:00:00,2017.000000,8.000000,8.000000,10.000000,27.000000,2017-05-06 18:00:00
50%,1.260982e+07,1.047460e+07,1.000000,0.000000,2019-02-01 00:00:00,2019.000000,16.000000,13.000000,29.000000,35.000000,2019-08-22 12:00:00
75%,1.276392e+07,1.054669e+07,2.000000,1.000000,2021-08-08 00:00:00,2021.000000,23.000000,17.000000,40.000000,47.000000,2022-11-05 12:00:00
max,1.292206e+07,1.061630e+07,30.000000,71.000000,2024-12-31 00:00:00,2024.000000,31.000000,23.000000,59.000000,2021.000000,2025-01-30 00:00:00
std,4.981276e+06,2.978044e+06,0.675795,1.337908,NaN,2.660038,8.779040,5.517070,17.414657,14.599189,NaN


In [22]:
# Cuenta la cantidad de valores nulos en cada columna.
Actor_vial.isnull().sum()

Codigo_Accidentado                     0
Codigo_Accidente                       0
Formulario                             0
Codigo_Vehiculo                    12686
Ccodigo_Victima                        0
FechaAcc                               0
AnnoAcc                                0
mesAcc                                 0
DD_Mes_Acc                             0
Dia_Semana_Acc                         0
hourAcc                                0
minuAcc                                0
Localidad                              0
Edad                               13337
Sexo                                5459
Gravedad_Indicador_Tradicional      4435
Muerte_Posterior                  325817
Fecha_CambioGravedad              606235
Gravedad_Indicador_30d              4435
Condicion                              0
Condicion_Especifica                   0
Tipo_SITP                         547619
Con_Bicicleta                     560904
Con_Carga                         528859
Con_Embriaguez  

In [23]:
# Cuenta el número de registros duplicados.
Actor_vial.duplicated().sum()

np.int64(0)

# Información 4. Diccionario

El diccionario describe campos de la fuente; no contiene casos de entrenamiento.
Se revisan sus definiciones, estructura, tipos y valores faltantes sin modificarlo.


In [24]:
display(Diccionario.head())
Diccionario.info(verbose=True, show_counts=True)
display(Diccionario.isna().sum().rename('Valores nulos').to_frame())


,Hoja,Atributo,Descripción
0,Siniestros,Codigo_Accidente,Clave primaria del ipat
1,Siniestros,Formulario,Número del Informe Policial de Accidente de Tr...
2,Siniestros,Longitud,Coordenada geográfica que indica la distancia ...
3,Siniestros,Latitud,Coordenada geográfica que indica la distancia ...
4,Siniestros,Direccion,Dirección completa donde ocurrió el siniestro ...


<class 'pandas.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Hoja         110 non-null    str  
 1   Atributo     110 non-null    str  
 2   Descripción  110 non-null    str  
dtypes: str(3)
memory usage: 14.8 KB


,Valores nulos
Hoja,0
Atributo,0
Descripción,0


## Resumen de la estructura de los conjuntos de datos

En esta sección se presenta un resumen general de los conjuntos de datos utilizados en el proyecto. Se identifica la cantidad de registros (filas) y variables (columnas) de cada hoja del archivo fuente, lo que permite dimensionar el volumen de información disponible para el análisis.

Adicionalmente, se revisan los tipos de datos de cada variable con el fin de identificar si corresponden a valores numéricos, categóricos, de texto o de fecha. Esta validación es fundamental para detectar posibles inconsistencias y definir las transformaciones necesarias durante la fase de preparación de los datos.

In [25]:
# Incluye las cuatro hojas, también el diccionario de variables.
resumen_hojas = pd.DataFrame([
    {'Hoja': nombre, 'Filas': tabla.shape[0], 'Columnas': tabla.shape[1]}
    for nombre, tabla in hojas.items()
])
display(resumen_hojas)


,Hoja,Filas,Columnas
0,Siniestros,278614,40
1,Vehiculos,523168,32
2,Actor_vial,608155,44
3,Diccionario,110,3


In [26]:
# display explícito: se muestran los tipos de TODAS las hojas, no solo la última.
for nombre, tabla in hojas.items():
    print(f'\nTipos de datos — {nombre}')
    with pd.option_context('display.max_rows', None):
        display(tabla.dtypes.rename('Tipo de dato pandas').rename_axis('Columna').to_frame())



Tipos de datos — Siniestros


,Tipo de dato pandas
Columna,
Codigo_Accidente,int64
Formulario,str
Longitud,float64
Latitud,float64
Direccion,str
Fecha_Acc,datetime64[us]
AA_Acc,int64
MM_Acc,str
DD_Mes_Acc,int64



Tipos de datos — Vehiculos


,Tipo de dato pandas
Columna,
Codigo_Accidente,int64
Formulario,str
Fecha_Acc,datetime64[us]
AA_Acc,int64
Codigo_Vehiculo,float64
Clase,str
Servicio,str
Modalidad,str
Vehiculo_Viajaba_Clasificado,str



Tipos de datos — Actor_vial


,Tipo de dato pandas
Columna,
Codigo_Accidentado,int64
Codigo_Accidente,int64
Formulario,str
Codigo_Vehiculo,float64
Ccodigo_Victima,int64
FechaAcc,datetime64[us]
AnnoAcc,int64
mesAcc,str
DD_Mes_Acc,int64



Tipos de datos — Diccionario


,Tipo de dato pandas
Columna,
Hoja,str
Atributo,str
Descripción,str
